# Practice 2.2 · JSON Parse 存活率

> **对应知识点**：EP2 · KP2 · 跳过 Preamble
> **测什么**：一行 prompt 改动，能让下游 `json.loads()` 从"经常崩"变成"100% 稳定"
> **预计时间**：5 分钟

---

## 你会看到什么

3 版 prompt 让模型输出 JSON，各跑 5 次，用 `json.loads()` 试 parse，比较成功率。

**预期**：版本 A 大概率有几次 parse 失败（因为 preamble 或 markdown 代码块）；版本 C 应该 100% 成功。

**这就是"生产可用 prompt" vs "玩具 prompt"的差别**。


In [ ]:
# ============ 前置：安装 SDK + 配置 client ============
!pip install openai -q

from openai import OpenAI
from google.colab import userdata   # Colab；Modelscope 上用 os.environ 或直接填 key

# 推荐用 Secrets 存 key（不要硬编码）
API_KEY = userdata.get("DEEPSEEK_KEY")   # Colab Secrets
# 或 Modelscope 上：
# API_KEY = "sk-你的DeepSeek key"

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.deepseek.com/v1"
)

MODEL = "deepseek-chat"

def call(prompt, **kwargs):
    """简化的 chat 调用。默认 temperature=0 保证实验可重现。"""
    kwargs.setdefault("temperature", 0)
    kwargs.setdefault("max_tokens", 512)
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        **kwargs
    )
    return r.choices[0].message.content


## 3 版 Prompt · 逐步升级

In [ ]:
# 版本 A · 无禁令（原始）
PROMPT_A = """从下面的文本抽取姓名和年龄，输出 JSON。

文本：张三，32 岁，杭州人。
"""

# 版本 B · 明确禁 preamble
PROMPT_B = """从下面的文本抽取姓名和年龄，输出 JSON。
只输出 JSON 对象，不要 preamble、不要客套话、不要 markdown 代码块。

文本：张三，32 岁，杭州人。
"""

# 版本 C · 禁令 + XML 包裹 + Schema 描述
PROMPT_C = """从 <text> 里抽取信息，按下面 JSON schema 输出。

<schema>
{
  "name": string,
  "age": number
}
</schema>

只输出 JSON 对象。不要 preamble，不要 markdown 代码块，不要 trailing text。

<text>
张三，32 岁，杭州人。
</text>
"""

## 跑测试 · 每版跑 5 次

In [ ]:
import json

def test_prompt(prompt, n=5):
    """跑 n 次，试 json.loads()，返回成功率 + 失败样本。"""
    success = 0
    failures = []
    for i in range(n):
        try:
            out = call(prompt)
            data = json.loads(out)
            success += 1
        except json.JSONDecodeError as e:
            failures.append(f"跑第 {i+1} 次失败：{out[:80]}...")
    return success, failures

print("=== 版本 A · 无禁令 ===")
s_a, f_a = test_prompt(PROMPT_A)
print(f"成功率：{s_a}/5")
for f in f_a: print(f)

print("\n=== 版本 B · 明确禁 preamble ===")
s_b, f_b = test_prompt(PROMPT_B)
print(f"成功率：{s_b}/5")
for f in f_b: print(f)

print("\n=== 版本 C · 禁令 + XML + Schema ===")
s_c, f_c = test_prompt(PROMPT_C)
print(f"成功率：{s_c}/5")
for f in f_c: print(f)

print(f"\n== 结论 ==")
print(f"A → B 提升：{s_b - s_a} 次")
print(f"B → C 提升：{s_c - s_b} 次")

# 硬 assert：C 应该显著优于 A
assert s_c >= s_a, "版本 C 应该 >= 版本 A"
print("\n✅ 通过：明确禁令 + XML + Schema 显著提升 JSON 输出可靠性")

## 一句心法

> **"一行禁令"是从"玩具 prompt"到"生产可用 prompt" 的分水岭**。
>
> 你觉得模型"总是输出 JSON"—— 但实际有 10-20% 的情况会加 preamble / 加代码块 / 加 trailing text。生产系统里这就是每 5 个请求 1 个崩。
>
> **版本 A 是几乎所有人第一次写的版本；版本 C 是几乎所有生产系统最终用的版本**。
